In [1]:
import cv2
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
import numpy as np

In [ ]:
img = cv2.imread('/content/drive/MyDrive/Visão Computacional Guia Completo/Images/megan.jpg')
cv2_imshow(img)

In [3]:
img.shape

(337, 600, 3)

In [4]:
type(img)

numpy.ndarray

In [5]:
img_blob = cv2.dnn.blobFromImage(image=img,
                                 scalefactor=1.0/255,
                                 size=(img.shape[1], img.shape[0]))

In [6]:
img_blob.shape

(1, 3, 337, 600)

In [7]:
model = cv2.dnn.readNetFromCaffe('/content/drive/MyDrive/Visão Computacional Guia Completo/Weights/pose_deploy_linevec_faster_4_stages.prototxt',
                                 '/content/drive/MyDrive/Visão Computacional Guia Completo/Weights/pose_iter_160000.caffemodel')

In [8]:
model.getLayerNames()

('conv1_1',
 'relu1_1',
 'conv1_2',
 'relu1_2',
 'pool1_stage1',
 'conv2_1',
 'relu2_1',
 'conv2_2',
 'relu2_2',
 'pool2_stage1',
 'conv3_1',
 'relu3_1',
 'conv3_2',
 'relu3_2',
 'conv3_3',
 'relu3_3',
 'conv3_4',
 'relu3_4',
 'pool3_stage1',
 'conv4_1',
 'relu4_1',
 'conv4_2',
 'relu4_2',
 'conv4_3_CPM',
 'relu4_3_CPM',
 'conv4_4_CPM',
 'relu4_4_CPM',
 'conv5_1_CPM_L1',
 'relu5_1_CPM_L1',
 'conv5_1_CPM_L2',
 'relu5_1_CPM_L2',
 'conv5_2_CPM_L1',
 'relu5_2_CPM_L1',
 'conv5_2_CPM_L2',
 'relu5_2_CPM_L2',
 'conv5_3_CPM_L1',
 'relu5_3_CPM_L1',
 'conv5_3_CPM_L2',
 'relu5_3_CPM_L2',
 'conv5_4_CPM_L1',
 'relu5_4_CPM_L1',
 'conv5_4_CPM_L2',
 'relu5_4_CPM_L2',
 'conv5_5_CPM_L1',
 'conv5_5_CPM_L2',
 'concat_stage2',
 'Mconv1_stage2_L1',
 'Mrelu1_stage2_L1',
 'Mconv1_stage2_L2',
 'Mrelu1_stage2_L2',
 'Mconv2_stage2_L1',
 'Mrelu2_stage2_L1',
 'Mconv2_stage2_L2',
 'Mrelu2_stage2_L2',
 'Mconv3_stage2_L1',
 'Mrelu3_stage2_L1',
 'Mconv3_stage2_L2',
 'Mrelu3_stage2_L2',
 'Mconv4_stage2_L1',
 'Mrelu4_sta

In [9]:
len(model.getLayerNames())

127

In [10]:
model.setInput(img_blob)
output = model.forward()

In [11]:
output.shape #imagens, pontos, eixos x e y

(1, 44, 43, 75)

In [12]:
pos_larg = output.shape[3]
pos_alt = output.shape[2]

In [13]:
pos_alt, pos_larg

(43, 75)

In [14]:
num_pontos = 15
pontos = []
threshold = 0.1

for i in range(num_pontos):
  mapa_confianca = output[0, i, :, :]
  _, confianca, _, loc = cv2.minMaxLoc(mapa_confianca)
  x = int((img.shape[1] * loc[0]) / pos_larg)
  y = int((img.shape[0] * loc[1]) / pos_alt)
  if confianca > threshold:
    cv2.circle(img, (x, y), 5, (0, 255, 0), thickness= -1)
    cv2.putText(img,'{}'.format(i), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255))
    pontos.append((x, y))
  else:
    pontos.append(None)

In [15]:
pontos

[(224, 23),
 (248, 62),
 (232, 94),
 (208, 125),
 (176, 148),
 (280, 62),
 (320, 39),
 (360, 15),
 (288, 180),
 (320, 250),
 (320, 321),
 (312, 164),
 (336, 250),
 (296, 305),
 (280, 125)]

In [ ]:
plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

In [17]:
ponto_conexoes = [[0, 1], [1, 2], [2, 3], [3, 4], [1, 5], [5, 6], [6, 7], [1, 14],
                  [14, 8], [8, 9], [9, 10], [14, 11], [11, 12], [12, 13]]

In [18]:
ponto_conexoes

[[0, 1],
 [1, 2],
 [2, 3],
 [3, 4],
 [1, 5],
 [5, 6],
 [6, 7],
 [1, 14],
 [14, 8],
 [8, 9],
 [9, 10],
 [14, 11],
 [11, 12],
 [12, 13]]

In [19]:
for conexao in ponto_conexoes:
  parteA = conexao[0]
  parteB = conexao[1]
  if pontos[parteA] and pontos[parteB]:
    cv2.line(img, pontos[parteA], pontos[parteB], (255, 0, 0))

In [ ]:
plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

In [ ]:
img = cv2.imread('/content/drive/MyDrive/Images/player.jpg')
cv2_imshow(img)

In [22]:
img_blob = cv2.dnn.blobFromImage(image=img,
                                 scalefactor=1.0/255,
                                 size=(img.shape[1], img.shape[0]))

In [23]:
model.setInput(img_blob)
output = model.forward()

In [24]:
pos_larg = output.shape[3]
pos_alt = output.shape[2]

In [25]:
num_pontos = 15
pontos = []
threshold = 0.1

for i in range(num_pontos):
  mapa_confianca = output[0, i, :, :]
  _, confianca, _, loc = cv2.minMaxLoc(mapa_confianca)
  x = int((img.shape[1] * loc[0]) / pos_larg)
  y = int((img.shape[0] * loc[1]) / pos_alt)
  if confianca > threshold:
    cv2.circle(img, (x, y), 5, (0, 255, 0), thickness= -1)
    cv2.putText(img,'{}'.format(i), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255))
    cv2.putText(img, '{}-{}'.format(loc[0], loc[1]), (x, y + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255))
    pontos.append((x, y))
  else:
    pontos.append(None)

In [ ]:
plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

In [27]:
def verification(pontos):
  cabeca, pulso_dir, pulso_esq = 0, 0, 0
  for i, loc in enumerate(pontos):
    if i == 0:
      cabeca = loc[1]
    elif i == 4:
      pulso_dir = loc[1]
    elif i == 7:
      pulso_esq = loc[1]
    if pulso_dir < cabeca and pulso_esq < cabeca:
      return True
    else:
      return False

In [28]:
verification(pontos)

True

In [29]:
video = '/content/drive/MyDrive/Visão Computacional Guia Completo/Videos/gesture1.mp4'
captura = cv2.VideoCapture(video)
conectado, frame = captura.read()

In [30]:
conectado

True

In [31]:
frame

array([[[148, 128,  98],
        [148, 128,  98],
        [148, 128,  98],
        ...,
        [152, 135, 104],
        [152, 135, 104],
        [152, 135, 104]],

       [[148, 128,  98],
        [148, 128,  98],
        [148, 128,  98],
        ...,
        [152, 135, 104],
        [152, 135, 104],
        [152, 135, 104]],

       [[148, 128,  98],
        [148, 128,  98],
        [148, 128,  98],
        ...,
        [152, 135, 104],
        [152, 135, 104],
        [152, 135, 104]],

       ...,

       [[  0,  74,   1],
        [  0,  74,   1],
        [  0,  74,   1],
        ...,
        [  0,  75,   0],
        [  0,  75,   0],
        [  0,  75,   0]],

       [[  0,  74,   1],
        [  0,  74,   1],
        [  0,  74,   1],
        ...,
        [  0,  75,   0],
        [  0,  75,   0],
        [  0,  75,   0]],

       [[  0,  74,   1],
        [  0,  74,   1],
        [  0,  74,   1],
        ...,
        [  0,  75,   0],
        [  0,  75,   0],
        [  0,  75,   0]]

In [32]:
resultado = '/content/drive/MyDrive/Visão Computacional Guia Completo/Videos/gesture1_result.mp4'
salvar_video = cv2.VideoWriter(resultado, cv2.VideoWriter_fourcc(* 'XVID'), 10, (frame.shape[1], frame.shape[0]))

In [33]:
threshold = 0.1

while cv2.waitKey(1) < 0:
  conectado, frame = captura.read()

  if not conectado:
    break

  img_blob = cv2.dnn.blobFromImage(image=frame,
                                 scalefactor=1.0/255,
                                 size=(256,256))
  model.setInput(img_blob)
  output = model.forward()
  pos_larg = output.shape[2]
  pos_alt = output.shape[3]

  num_pontos = 15
  pontos = []

  for i in range(num_pontos):
    mapa_confianca = output[0, i, :, :]
    _, confianca, _, loc = cv2.minMaxLoc(mapa_confianca)
    x = int((frame.shape[1] * loc[0]) / pos_larg)
    y = int((frame.shape[0] * loc[1]) / pos_alt)
    if confianca > threshold:
      cv2.circle(frame, (x, y), 5, (0, 255, 0), thickness= -1)
      cv2.putText(frame,'{}'.format(i), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255))
      pontos.append((x, y))
    else:
      pontos.append(None)

  for conexao in ponto_conexoes:
    parteA = conexao[0]
    parteB = conexao[1]
    if pontos[parteA] and pontos[parteB]:
      cv2.line(frame, pontos[parteA], pontos[parteB], (255, 0, 0))

  if verification(pontos) == True:
    cv2.putText(frame, 'Completo!', (50, 200), cv2.FONT_HERSHEY_COMPLEX, 3, (0, 0, 255))

  cv2_imshow(frame)
  salvar_video.write(frame)
salvar_video.release()
print('Processamento Concluído')

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
img = cv2.imread('/content/drive/MyDrive/Images/jump.jpg')
cv2_imshow(img)

In [41]:
img_blob = cv2.dnn.blobFromImage(image=img,
                                 scalefactor=1.0/255,
                                 size=(img.shape[1], img.shape[0]))

model.setInput(img_blob)
output = model.forward()

pos_larg = output.shape[3]
pos_alt = output.shape[2]

num_pontos = 15
pontos = []
threshold = 0.1

for i in range(num_pontos):
  mapa_confianca = output[0, i, :, :]
  _, confianca, _, loc = cv2.minMaxLoc(mapa_confianca)
  x = int((img.shape[1] * loc[0]) / pos_larg)
  y = int((img.shape[0] * loc[1]) / pos_alt)
  if confianca > threshold:
    cv2.circle(img, (x, y), 5, (0, 255, 0), thickness= -1)
    cv2.putText(img,'{}'.format(i), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255))
    cv2.putText(img, '{}-{}'.format(loc[0], loc[1]), (x, y + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255))
    pontos.append((x, y))
  else:
    pontos.append(None)

In [ ]:
plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

In [50]:
def verification2(pontos):
  perna_dir, perna_esq = 0, 0
  quadril_dir, quadril_esq = 0, 0
  for i, loc in enumerate(pontos):
    if i == 11:
      quadril_esq = loc[0]
    elif i == 8:
      quadril_dir = loc[0]
    elif i == 13:
      perna_esq = loc[0]
    elif i == 10:
      perna_dir = loc[0]
  if (perna_esq > quadril_esq) and (perna_dir < quadril_dir):
    return True
  else:
    return False

In [51]:
verification2(pontos)

True